In [1]:
pip install psycopg2-binary -q

Note: you may need to restart the kernel to use updated packages.


In [2]:
import psycopg2

# 데이터베이스 연결 정보 (test_db로 변경)
DB_CONFIG = {
    "dbname": "test_db", 
    "user": "postgres",
    "password": "psql",
    "host": "172.16.174.129",
    "port": "5432"
}

def run_db_test():
    conn = None
    try:
        # 1. 데이터베이스 연결
        conn = psycopg2.connect(**DB_CONFIG)
        cur = conn.cursor()
        print(" [성공] test_db 연결 완료!")

        # 2. 커스텀 함수 생성 (NVL, NVL2)
        cur.execute("""
            CREATE OR REPLACE FUNCTION nvl(anyelement, anyelement) 
            RETURNS anyelement AS $$
            BEGIN RETURN COALESCE($1, $2); END;
            $$ LANGUAGE plpgsql;

            CREATE OR REPLACE FUNCTION nvl2(anyelement, anyelement, anyelement) 
            RETURNS anyelement AS $$
            BEGIN
                RETURN CASE WHEN $1 IS NOT NULL THEN $2 ELSE $3 END;
            END;
            $$ LANGUAGE plpgsql;
        """)

        # 3. 테이블 생성 및 NULL 데이터를 포함한 INSERT
        cur.execute("DROP TABLE IF EXISTS user_test;")
        cur.execute("CREATE TABLE user_test (id SERIAL, name VARCHAR(20), email VARCHAR(50));")
        
        # NULL을 직접 INSERT 하는 쿼리
        test_data = [
            ('홍길동', 'hong@test.com'),
            (None, 'no_name@test.com'),    # name이 NULL인 경우
            ('이순신', None),             # email이 NULL인 경우
            (None, None)                 # 둘 다 NULL인 경우
        ]
        
        for item in test_data:
            cur.execute("INSERT INTO user_test (name, email) VALUES (%s, %s);", item)
        
        conn.commit()
        print(" [성공] NULL 데이터를 포함한 데이터 삽입 완료!")

        # 4. 함수 검증을 위한 SELECT
        print("\n--- 검증 결과 ---")
        print(f"{'이름(NULL포함)':<12} | {'이메일':<20} | {'NVL 결과':<10} | {'NVL2 결과(이메일여부)':<20} | {'COALESCE 결과 (전체)': <20}")
        print("-" * 110)
        
        cur.execute("""
            SELECT 
                name, email,
                nvl(name, '이름없음'), 
                nvl2(email, '이메일있음', '이메일없음') ,
                COALESCE(name, email, '모든정보없음') AS result_coalesce
            FROM user_test;
        """)

        # 핵심은 nvl 은 name 이 (1번 인자가) NULL 이면 이름없음 (2번 인자)
        # nvl2 는 email 이 (1번 인자가) 존재하면 이메일 있음 (2번 인자) , 반대로 NULL 이면 이메일 없음 (3번 인자)
        # COALESCE 는 NULL 이 아닌 첫 번째 값을 반환
        
        for row in cur.fetchall():
            print(f"{str(row[0]):<12} | {str(row[1]):<22} | {str(row[2]):<12} | {str(row[3]):<20} | {str(row[4]):<20}")

        cur.close()
        conn.close()

    except Exception as e:
        print(f" [오류] {e}")
        if conn: conn.rollback()

if __name__ == "__main__":
    run_db_test()

 [성공] test_db 연결 완료!
 [성공] NULL 데이터를 포함한 데이터 삽입 완료!

--- 검증 결과 ---
이름(NULL포함)   | 이메일                  | NVL 결과     | NVL2 결과(이메일여부)       | COALESCE 결과 (전체)    
--------------------------------------------------------------------------------------------------------------
홍길동          | hong@test.com          | 홍길동          | 이메일있음                | 홍길동                 
None         | no_name@test.com       | 이름없음         | 이메일있음                | no_name@test.com    
이순신          | None                   | 이순신          | 이메일없음                | 이순신                 
None         | None                   | 이름없음         | 이메일없음                | 모든정보없음              


In [3]:
# 데이터베이스 연결 정보 (test_db로 변경)
DB_CONFIG = {
    "dbname": "test_db", 
    "user": "postgres",
    "password": "psql",
    "host": "172.16.174.129",
    "port": "5432"
}

def run_db_test():
    conn = None
    try:
        # 1. 데이터베이스 연결
        conn = psycopg2.connect(**DB_CONFIG)
        cur = conn.cursor()
        print(" [성공] test_db 연결 완료!")

        # 2. 커스텀 함수 생성 (NVL, NVL2)
        cur.execute("""
            CREATE OR REPLACE FUNCTION nvl(anyelement, anyelement) 
            RETURNS anyelement AS $$
            BEGIN RETURN COALESCE($1, $2); END;
            $$ LANGUAGE plpgsql;

            CREATE OR REPLACE FUNCTION nvl2(anyelement, anyelement, anyelement) 
            RETURNS anyelement AS $$
            BEGIN
                RETURN CASE WHEN $1 IS NOT NULL THEN $2 ELSE $3 END;
            END;
            $$ LANGUAGE plpgsql;

            CREATE OR REPLACE FUNCTION ifnull(anyelement, anyelement) 
            RETURNS anyelement AS $$
            BEGIN
                RETURN COALESCE($1, $2);
            END;
            $$ LANGUAGE plpgsql;
        """)

        # 3. 테이블 생성 및 NULL 데이터를 포함한 INSERT
        cur.execute("DROP TABLE IF EXISTS user_test;")
        cur.execute("CREATE TABLE user_test (id SERIAL, name VARCHAR(20), email VARCHAR(50) , phone VARCHAR(50));")
        
        # NULL을 직접 INSERT 하는 쿼리
        test_data = [
            ('홍길동', 'hong@test.com' , '010-1234-5678'),
            (None, 'no_name@test.com' , '010-1234-6678'),      # name이 NULL인 경우
            ('이순신', None , '010-1234-7678'),               # email이 NULL인 경우
            (None, None  , '010-1234-8678') ,                # 둘 다 NULL인 경우
            (None , None , None)                             # 셋 다 NULL인 경우
        ]
        
        for item in test_data:
            cur.execute("INSERT INTO user_test (name, email , phone) VALUES (%s, %s , %s);", item)
        
        conn.commit()
        print(" [성공] NULL 데이터를 포함한 데이터 삽입 완료!")

        # 4. 함수 검증을 위한 SELECT
        print("\n--- 검증 결과 ---")
        print(f"{'이름(NULL포함)':<12} | {'이메일':<20} | {'NVL 결과':<10} | {'NVL2 결과(이메일여부)':<20} | {'IFNULL 결과 (번호여부)': <20} | {'COALESCE 결과 (전체)': <20}")
        print("-" * 140)
        
        cur.execute("""
            SELECT 
                name, email,
                nvl(name, '이름없음'), 
                nvl2(email, '이메일있음', '이메일없음') ,
                ifnull(phone , '휴대폰 없음') ,
                COALESCE(name, email, phone, '모든정보없음') AS result_coalesce
            FROM user_test;
        """)

        # 핵심은 nvl 은 name 이 (1번 인자가) NULL 이면 이름없음 (2번 인자)
        # nvl2 는 email 이 (1번 인자가) 존재하면 이메일 있음 (2번 인자) , 반대로 NULL 이면 이메일 없음 (3번 인자)
        # COALESCE 는 NULL 이 아닌 첫 번째 값을 반환
        
        for row in cur.fetchall():
            print(f"{str(row[0]):<12} | {str(row[1]):<22} | {str(row[2]):<12} | {str(row[3]):<20} | {str(row[4]):<25} | {str(row[5]):<20}")

        cur.close()
        conn.close()

    except Exception as e:
        print(f" [오류] {e}")
        if conn: conn.rollback()

if __name__ == "__main__":
    run_db_test()

 [성공] test_db 연결 완료!
 [성공] NULL 데이터를 포함한 데이터 삽입 완료!

--- 검증 결과 ---
이름(NULL포함)   | 이메일                  | NVL 결과     | NVL2 결과(이메일여부)       | IFNULL 결과 (번호여부)     | COALESCE 결과 (전체)    
--------------------------------------------------------------------------------------------------------------------------------------------
홍길동          | hong@test.com          | 홍길동          | 이메일있음                | 010-1234-5678             | 홍길동                 
None         | no_name@test.com       | 이름없음         | 이메일있음                | 010-1234-6678             | no_name@test.com    
이순신          | None                   | 이순신          | 이메일없음                | 010-1234-7678             | 이순신                 
None         | None                   | 이름없음         | 이메일없음                | 010-1234-8678             | 010-1234-8678       
None         | None                   | 이름없음         | 이메일없음                | 휴대폰 없음                    | 모든정보없음              


In [4]:
# 사용자 지정 함수 지우기

import psycopg2

DB_CONFIG = {
    "dbname": "test_db", 
    "user": "postgres",
    "password": "psql",
    "host": "172.16.174.129",
    "port": "5432"
}

def drop_custom_functions():
    conn = None
    try:
        conn = psycopg2.connect(**DB_CONFIG)
        cur = conn.cursor()
        
        # 커스텀 함수 삭제 쿼리 실행
        drop_sql = """
        DROP FUNCTION IF EXISTS nvl(anyelement, anyelement);
        DROP FUNCTION IF EXISTS nvl2(anyelement, anyelement, anyelement);
        DROP FUNCTION IF EXISTS ifnull(anyelement, anyelement);
        """
        cur.execute(drop_sql)
        conn.commit()
        
        print(" [성공] nvl, nvl2, ifnull 커스텀 함수를 지웠습니다.")
        
        cur.close()
        conn.close()

    except Exception as e:
        print(f" [오류] {e}")
        if conn: conn.rollback()

if __name__ == "__main__":
    drop_custom_functions()

 [성공] nvl, nvl2, ifnull 커스텀 함수를 지웠습니다.
